# Benchmark 3 — Gradient conjugué vs Newton vs BFGS

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from joblib import Parallel, delayed
from tqdm.auto import tqdm

from src.test_functions import create_system, quadraticn_, Rosenbrock, multitrous2_

In [ ]:
METHODS = ["Newton", "CG-PR", "BFGS"]
METHOD_COLORS = {"Newton": "#2ca02c", "CG-PR": "#1f77b4", "BFGS": "#d62728"}
METHOD_LS = {"Newton": "-", "CG-PR": "--", "BFGS": "-."}

DIMS = [2, 5, 10, 50, 100]
KAPPAS = [5, 10, 50, 100, 500]
N_SEEDS = 20

ROSEN_GAMMAS = [1, 10, 100]
MULTI_GAMMAS = [1, 2, 4, 8]
N_X0 = 20

eps = 1e-6
max_iter = 10000

OUT = "../figures/benchmark3"
os.makedirs(OUT, exist_ok=True)

plt.rcParams["mathtext.fontset"] = "cm"
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11

In [ ]:
class FuncCounter:
    def __init__(self, f):
        self.f = f
        self.count = 0

    def __call__(self, x):
        self.count += 1
        return self.f(x)


def grad_quadraticn(x, A, b):
    return A @ x - b


def grad_rosenbrock(x, gamma):
    return np.array([
        2 * (x[0] - 1) + 4 * gamma * x[0] * (x[0]**2 - x[1]),
        -2 * gamma * (x[0]**2 - x[1]),
    ])


def grad_multitrous2(x, gamma):
    return np.array([
        -40 * x[0] * np.sin(x[0]**2) + 2 * x[0],
        -40 * x[1] * np.sin(x[1]**2) + 2 * gamma * x[1],
    ])

In [ ]:
def armijo(xk, fk, f, dk, gk_dot_dk, alpha=0.25, beta=0.5):
    eta = 1.0
    for _ in range(60):
        xt = xk + eta * dk
        ft = f(xt)
        if ft <= fk + alpha * eta * gk_dot_dk:
            return eta, ft, xt
        eta *= beta
    return eta, f(xk + eta * dk), xk + eta * dk

In [ ]:
def _run_quad(mi, di, ki, si, method, dim, kappa, seed, eps, max_iter):
    np.seterr(all="ignore")

    A, b = create_system(dim, cond=kappa, seed=seed)
    x_star = np.linalg.solve(A, b)
    x0 = np.zeros(dim)

    f_raw = lambda x, A=A, b=b: (x.T @ A @ x) / 2 - b.T @ x
    gf = lambda x, A=A, b=b: A @ x - b
    fc = FuncCounter(f_raw)

    errors = [float(np.linalg.norm(x0 - x_star))]
    xk = x0.copy()
    converged = False

    if method == "Newton":
        A_inv = np.linalg.inv(A)
        fk = fc(xk)
        for it in range(1, max_iter + 1):
            gk = gf(xk)
            if np.linalg.norm(gk) < eps:
                converged = True
                break
            dk = -A_inv @ gk
            xk = xk + dk
            fk = fc(xk)
            errors.append(float(np.linalg.norm(xk - x_star)))
        n_iter = it - 1 if converged else max_iter

    elif method == "CG-PR":
        fk = fc(xk)
        gk_prev = None
        dk = None
        n_iter = 0
        for it in range(1, max_iter + 1):
            gk = gf(xk)
            gn = np.linalg.norm(gk)
            if gn < eps:
                converged = True
                n_iter = it - 1
                break
            if dk is None:
                dk = -gk
            else:
                gpg = np.dot(gk_prev, gk_prev)
                if gpg < 1e-30:
                    dk = -gk
                else:
                    beta_pr = np.dot(gk, gk - gk_prev) / gpg
                    dk = -gk + beta_pr * dk
            gd = np.dot(gk, dk)
            if gd >= 0:
                dk = -gk
                gd = np.dot(gk, dk)
            eta, fk, xk = armijo(xk, fk, fc, dk, gd)
            errors.append(float(np.linalg.norm(xk - x_star)))
            gk_prev = gk.copy()
        if not converged:
            n_iter = max_iter

    else: # BFGS
        n = dim
        Hk = np.eye(n)
        fk = fc(xk)
        gk = gf(xk)
        n_iter = 0
        for it in range(1, max_iter + 1):
            if np.linalg.norm(gk) < eps:
                converged = True
                n_iter = it - 1
                break
            dk = -Hk @ gk
            gd = np.dot(gk, dk)
            if gd >= 0:
                dk = -gk
                gd = np.dot(gk, dk)
                Hk = np.eye(n)
            eta, fk1, xk1 = armijo(xk, fk, fc, dk, gd)
            sk = xk1 - xk
            gk1 = gf(xk1)
            yk = gk1 - gk
            sy = np.dot(sk, yk)
            if sy > 1e-12:
                rho = 1.0 / sy
                I = np.eye(n)
                V = I - rho * np.outer(sk, yk)
                Hk = V @ Hk @ V.T + rho * np.outer(sk, sk)
            xk, fk, gk = xk1, fk1, gk1
            errors.append(float(np.linalg.norm(xk - x_star)))
        if not converged:
            n_iter = max_iter

    final_err = float(np.linalg.norm(xk - x_star))
    return (mi, di, ki, si, [n_iter, fc.count, final_err, int(converged)], errors)

In [ ]:
CACHE_QUAD = "../results_b3_quad.npz"

if os.path.exists(CACHE_QUAD):
    _ld = np.load(CACHE_QUAD, allow_pickle=True)
    results_quad = _ld["metrics"]
    traj_quad = _ld["trajectories"]
    print(f"Charge depuis le cache : metrics {results_quad.shape}")
else:
    tasks = [
        (mi, di, ki, si, method, dim, kappa, si, eps, max_iter)
        for mi, method in enumerate(METHODS)
        for di, dim in enumerate(DIMS)
        for ki, kappa in enumerate(KAPPAS)
        for si in range(N_SEEDS)
    ]
    results_quad = np.full((len(METHODS), len(DIMS), len(KAPPAS), N_SEEDS, 4), np.nan)
    traj_quad_dict = {}

    for mi, di, ki, si, vals, errs in tqdm(
        Parallel(n_jobs=-1, return_as="generator")(
            delayed(_run_quad)(*t) for t in tasks
        ),
        total=len(tasks),
        desc="quadratic regime",
    ):
        results_quad[mi, di, ki, si] = vals
        traj_quad_dict[(mi, di, ki, si)] = errs

    max_traj_len = max(len(v) for v in traj_quad_dict.values())
    traj_quad = np.full(
        (len(METHODS), len(DIMS), len(KAPPAS), N_SEEDS, max_traj_len), np.nan
    )
    for (mi, di, ki, si), errs in traj_quad_dict.items():
        traj_quad[mi, di, ki, si, :len(errs)] = errs

    np.savez(CACHE_QUAD, metrics=results_quad, trajectories=traj_quad)
    print(f"Sauvegarde : metrics {results_quad.shape}, traj {traj_quad.shape}")

In [ ]:
def _run_general(mi, fi, gi, xi, method, func_name, gamma, x0_0, x0_1, eps, max_iter):
    np.seterr(all="ignore")

    if func_name == "rosenbrock":
        f_raw = lambda x, g=gamma: Rosenbrock(x, gamma=g)
        gf = lambda x, g=gamma: grad_rosenbrock(x, g)
        x_star = np.array([1.0, 1.0])
    else:
        f_raw = lambda x, g=gamma: multitrous2_(x, gamma=g)
        gf = lambda x, g=gamma: grad_multitrous2(x, g)
        x_star = None

    fc = FuncCounter(f_raw)
    xk = np.array([x0_0, x0_1])
    fk = fc(xk)
    gk_prev = None
    dk = None
    converged = False
    n = 2

    if x_star is not None:
        errors = [float(np.linalg.norm(xk - x_star))]
    else:
        errors = []

    if method == "CG-PR":
        for it in range(1, max_iter + 1):
            gk = gf(xk)
            if not np.all(np.isfinite(gk)) or not np.isfinite(fk):
                break
            if np.linalg.norm(gk) < eps:
                converged = True
                break
            if dk is None:
                dk = -gk
            else:
                gpg = np.dot(gk_prev, gk_prev)
                if gpg < 1e-30:
                    dk = -gk
                else:
                    beta_pr = np.dot(gk, gk - gk_prev) / gpg
                    dk = -gk + beta_pr * dk
            gd = np.dot(gk, dk)
            if gd >= 0:
                dk = -gk
                gd = np.dot(gk, dk)
            eta, fk, xk = armijo(xk, fk, fc, dk, gd)
            gk_prev = gk.copy()
            if x_star is not None:
                errors.append(float(np.linalg.norm(xk - x_star)))
        n_iter = it if converged else (it if it < max_iter else max_iter)

    else: # BFGS
        Hk = np.eye(n)
        gk = gf(xk)
        for it in range(1, max_iter + 1):
            if not np.all(np.isfinite(gk)) or not np.isfinite(fk):
                break
            if np.linalg.norm(gk) < eps:
                converged = True
                break
            dk = -Hk @ gk
            gd = np.dot(gk, dk)
            if gd >= 0:
                dk = -gk
                gd = np.dot(gk, dk)
                Hk = np.eye(n)
            eta, fk1, xk1 = armijo(xk, fk, fc, dk, gd)
            sk = xk1 - xk
            gk1 = gf(xk1)
            yk = gk1 - gk
            sy = np.dot(sk, yk)
            if sy > 1e-12:
                rho = 1.0 / sy
                I = np.eye(n)
                V = I - rho * np.outer(sk, yk)
                Hk = V @ Hk @ V.T + rho * np.outer(sk, sk)
            xk, fk, gk = xk1, fk1, gk1
            if x_star is not None:
                errors.append(float(np.linalg.norm(xk - x_star)))
        n_iter = it if converged else (it if it < max_iter else max_iter)

    f_final = float(f_raw(xk)) if np.all(np.isfinite(xk)) else np.nan
    final_err = float(np.linalg.norm(xk - x_star)) if x_star is not None else np.nan
    return (mi, fi, gi, xi, [n_iter, fc.count, f_final, final_err, int(converged)], errors)

In [ ]:
CACHE_GEN = "../results_b3_gen.npz"

GEN_METHODS = ["CG-PR", "BFGS"]
FUNC_CONFIGS = {
    "rosenbrock": {"gammas": ROSEN_GAMMAS, "domain": (-2.0, 3.0)},
    "multitrous2": {"gammas": MULTI_GAMMAS, "domain": (-4.0, 4.0)},
}
FUNC_NAMES = list(FUNC_CONFIGS.keys())

rng = np.random.default_rng(42)
x0_gen = {}
for fname, cfg in FUNC_CONFIGS.items():
    lo, hi = cfg["domain"]
    x0_gen[fname] = rng.uniform(lo, hi, size=(N_X0, 2))

if os.path.exists(CACHE_GEN):
    _ld = np.load(CACHE_GEN, allow_pickle=True)
    results_gen_dict = {k: _ld[k] for k in _ld.files if k.startswith("metrics_")}
    traj_gen_dict = {k: _ld[k] for k in _ld.files if k.startswith("traj_")}
    print(f"Charge depuis le cache : {len(results_gen_dict)} configurations")
else:
    tasks = []
    for mi, method in enumerate(GEN_METHODS):
        for fi, fname in enumerate(FUNC_NAMES):
            gammas = FUNC_CONFIGS[fname]["gammas"]
            for gi, gamma in enumerate(gammas):
                for xi in range(N_X0):
                    x0 = x0_gen[fname][xi]
                    tasks.append(
                        (mi, fi, gi, xi, method, fname, gamma, x0[0], x0[1], eps, max_iter)
                    )

    raw_results = {}
    raw_trajs = {}
    for mi, fi, gi, xi, vals, errs in tqdm(
        Parallel(n_jobs=-1, return_as="generator")(
            delayed(_run_general)(*t) for t in tasks
        ),
        total=len(tasks),
        desc="general regime",
    ):
        raw_results[(mi, fi, gi, xi)] = vals
        raw_trajs[(mi, fi, gi, xi)] = errs

    results_gen_dict = {}
    traj_gen_dict = {}
    for fi, fname in enumerate(FUNC_NAMES):
        gammas = FUNC_CONFIGS[fname]["gammas"]
        n_g = len(gammas)
        arr = np.full((len(GEN_METHODS), n_g, N_X0, 5), np.nan)
        for mi in range(len(GEN_METHODS)):
            for gi in range(n_g):
                for xi in range(N_X0):
                    arr[mi, gi, xi] = raw_results[(mi, fi, gi, xi)]
        results_gen_dict[f"metrics_{fname}"] = arr

        if fname == "rosenbrock":
            max_len = 1
            for mi in range(len(GEN_METHODS)):
                for gi in range(n_g):
                    for xi in range(N_X0):
                        max_len = max(max_len, len(raw_trajs[(mi, fi, gi, xi)]))
            tarr = np.full((len(GEN_METHODS), n_g, N_X0, max_len), np.nan)
            for mi in range(len(GEN_METHODS)):
                for gi in range(n_g):
                    for xi in range(N_X0):
                        e = raw_trajs[(mi, fi, gi, xi)]
                        tarr[mi, gi, xi, :len(e)] = e
            traj_gen_dict[f"traj_{fname}"] = tarr

    np.savez(CACHE_GEN, **results_gen_dict, **traj_gen_dict)
    print("Regime general sauvegarde.")

In [ ]:
ITER, FEVAL, ERR, CONV = 0, 1, 2, 3
G_ITER, G_FEVAL, G_FVAL, G_ERR, G_CONV = 0, 1, 2, 3, 4

rq = results_quad
print("Quadratic metrics shape:", rq.shape)
print("Convergence rates (Newton/CG-PR/BFGS):",
      [f"{rq[mi, ..., CONV].mean():.1%}" for mi, m in enumerate(METHODS)])

In [ ]:
kappa_show = [50, 500]
fig, axes = plt.subplots(1, len(kappa_show), figsize=(7 * len(kappa_show), 5),
                         constrained_layout=True, sharey=True)
if len(kappa_show) == 1:
    axes = [axes]

for panel, kappa in enumerate(kappa_show):
    ax = axes[panel]
    ki = KAPPAS.index(kappa)
    width = 0.25
    positions_base = np.arange(len(DIMS))

    for mi, method in enumerate(METHODS):
        data = []
        for di in range(len(DIMS)):
            vals = rq[mi, di, ki, :, ITER]
            conv_mask = rq[mi, di, ki, :, CONV] > 0.5
            vals = vals[conv_mask]
            data.append(vals if len(vals) > 0 else np.array([np.nan]))

        pos = positions_base + (mi - 1) * width
        bp = ax.boxplot(data, positions=pos, widths=width * 0.85,
                        patch_artist=True, showfliers=False)
        for patch in bp["boxes"]:
            patch.set_facecolor(METHOD_COLORS[method])
            patch.set_alpha(0.7)
        for element in ["whiskers", "caps", "medians"]:
            for line in bp[element]:
                line.set_color("black")
                line.set_linewidth(0.8)

    ax.set_xticks(positions_base)
    ax.set_xticklabels([str(d) for d in DIMS])
    ax.set_xlabel("Dimension $n$")
    ax.set_yscale("log")
    ax.set_title(rf"$\kappa = {kappa}$")
    ax.grid(True, which="both", alpha=0.3, ls="--")
    if panel == 0:
        ax.set_ylabel("Nombre d'itérations")
        handles = [plt.Line2D([0], [0], color=METHOD_COLORS[m], lw=6, alpha=0.7)
                   for m in METHODS]
        ax.legend(handles, METHODS, loc="upper left", fontsize=10)

fig.suptitle("Effet de la dimension sur la convergence",
             fontsize=13, weight="bold")
fig.savefig(f"{OUT}/fig1_scaling_dim.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
dim_show = [10, 50]
fig, axes = plt.subplots(1, len(dim_show), figsize=(7 * len(dim_show), 5),
                         constrained_layout=True, sharey=True)
if len(dim_show) == 1:
    axes = [axes]

for panel, dim in enumerate(dim_show):
    ax = axes[panel]
    di = DIMS.index(dim)
    width = 0.25
    positions_base = np.arange(len(KAPPAS))

    for mi, method in enumerate(METHODS):
        data = []
        for ki in range(len(KAPPAS)):
            vals = rq[mi, di, ki, :, ITER]
            conv_mask = rq[mi, di, ki, :, CONV] > 0.5
            vals = vals[conv_mask]
            data.append(vals if len(vals) > 0 else np.array([np.nan]))

        pos = positions_base + (mi - 1) * width
        bp = ax.boxplot(data, positions=pos, widths=width * 0.85,
                        patch_artist=True, showfliers=False)
        for patch in bp["boxes"]:
            patch.set_facecolor(METHOD_COLORS[method])
            patch.set_alpha(0.7)
        for element in ["whiskers", "caps", "medians"]:
            for line in bp[element]:
                line.set_color("black")
                line.set_linewidth(0.8)

    ax.set_xticks(positions_base)
    ax.set_xticklabels([str(k) for k in KAPPAS])
    ax.set_xlabel(r"Conditionnement $\kappa$")
    ax.set_yscale("log")
    ax.set_title(f"$n = {dim}$")
    ax.grid(True, which="both", alpha=0.3, ls="--")
    if panel == 0:
        ax.set_ylabel("Nombre d'itérations")
        handles = [plt.Line2D([0], [0], color=METHOD_COLORS[m], lw=6, alpha=0.7)
                   for m in METHODS]
        ax.legend(handles, METHODS, loc="upper left", fontsize=10)

fig.suptitle("Effet du conditionnement sur la convergence",
             fontsize=13, weight="bold")
fig.savefig(f"{OUT}/fig2_scaling_kappa.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
di_conv = DIMS.index(10)

fig, axes = plt.subplots(1, len(KAPPAS), figsize=(4.2 * len(KAPPAS), 4.5),
                         constrained_layout=True, sharey=False)

for ki, kappa in enumerate(KAPPAS):
    ax = axes[ki]

    for mi, method in enumerate(METHODS):
        trajs = traj_quad[mi, di_conv, ki]
        lens = (~np.isnan(trajs)).sum(axis=1)
        max_common = int(np.median(lens))
        if max_common < 2:
            ax.axhline(y=np.nanmedian(trajs[:, 0]), color=METHOD_COLORS[method],
                       ls=METHOD_LS[method], lw=1.5, label=f"{method} (1 iter)")
            continue
        clipped = trajs[:, :max_common]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            med = np.nanmedian(clipped, axis=0)
            q25 = np.nanpercentile(clipped, 25, axis=0)
            q75 = np.nanpercentile(clipped, 75, axis=0)
        iters = np.arange(max_common)
        mask = med > 0
        ax.semilogy(iters[mask], med[mask], color=METHOD_COLORS[method],
                    ls=METHOD_LS[method], lw=1.8, label=method)
        ax.fill_between(iters[mask], q25[mask], q75[mask],
                        color=METHOD_COLORS[method], alpha=0.12)

    rho = (kappa - 1) / (kappa + 1)
    e0 = float(np.nanmedian(traj_quad[:, di_conv, ki, :, 0]))
    ref_len = int(np.nanmedian((~np.isnan(traj_quad[1, di_conv, ki])).sum(axis=1)))
    ref_iters = np.arange(min(ref_len, 500))
    ax.semilogy(ref_iters, e0 * rho**ref_iters, color="black", ls=":", lw=1.0,
                alpha=0.5, label=rf"$\rho^k$ ($\rho={rho:.3f}$)")

    ax.set_title(rf"$\kappa = {kappa}$")
    ax.set_xlabel("Itérations")
    if ki == 0:
        ax.set_ylabel(r"$\|x_k - x^\star\|_2$")
    ax.grid(True, which="major", alpha=0.3, ls="--")
    if ki == 0:
        ax.legend(fontsize=7.5, loc="upper right")

fig.suptitle(r"Convergence sur fonction quadratique ($n=10$)",
             fontsize=13, weight="bold")
fig.savefig(f"{OUT}/fig3_convergence_quad.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
di_ord = DIMS.index(10)
kappa_ord = [50, 500]

fig, axes = plt.subplots(len(kappa_ord), 2, figsize=(12, 5 * len(kappa_ord)),
                         constrained_layout=True)

for row, kappa in enumerate(kappa_ord):
    ki = KAPPAS.index(kappa)
    for col, (mi, method) in enumerate([(1, "CG-PR"), (2, "BFGS")]):
        ax = axes[row, col] if len(kappa_ord) > 1 else axes[col]
        all_ek = []
        all_ek1 = []
        for si in range(N_SEEDS):
            errs = traj_quad[mi, di_ord, ki, si]
            errs = errs[np.isfinite(errs)]
            errs = errs[errs > 1e-15]
            if len(errs) < 3:
                continue
            all_ek.extend(errs[:-1])
            all_ek1.extend(errs[1:])

        all_ek = np.array(all_ek)
        all_ek1 = np.array(all_ek1)
        if len(all_ek) < 3:
            ax.set_title(f"{method}, $\\kappa={kappa}$ (trop peu de points)")
            continue

        log_ek = np.log10(all_ek)
        log_ek1 = np.log10(all_ek1)

        ax.scatter(log_ek, log_ek1, s=8, alpha=0.4, color=METHOD_COLORS[method])

        mask = np.isfinite(log_ek) & np.isfinite(log_ek1)
        if mask.sum() >= 3:
            coeffs = np.polyfit(log_ek[mask], log_ek1[mask], 1)
            slope = coeffs[0]
            x_fit = np.linspace(log_ek[mask].min(), log_ek[mask].max(), 50)
            ax.plot(x_fit, np.polyval(coeffs, x_fit), "k-", lw=1.5,
                    label=rf"pente $= {slope:.2f}$")

        xlim = ax.get_xlim()
        x_ref = np.linspace(xlim[0], xlim[1], 50)
        ax.plot(x_ref, x_ref, ":", color="gray", lw=0.8, alpha=0.6, label="ordre 1")
        ax.plot(x_ref, 2 * x_ref - x_ref.mean(), "--", color="gray", lw=0.8,
                alpha=0.6, label="ordre 2")

        ax.set_xlabel(r"$\log_{10}\|e_k\|$")
        ax.set_ylabel(r"$\log_{10}\|e_{k+1}\|$")
        ax.set_title(rf"{method}, $\kappa={kappa}$")
        ax.legend(fontsize=8, loc="upper left")
        ax.grid(True, alpha=0.3, ls="--")
        ax.set_aspect("equal", adjustable="datalim")

fig.suptitle(r"Ordre de convergence empirique ($n=10$, quadratique)",
             fontsize=13, weight="bold")
fig.savefig(f"{OUT}/fig4_convergence_order.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
di_cost = DIMS.index(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

ax = axes[0]
width = 0.22
positions_base = np.arange(len(KAPPAS))
for mi, method in enumerate(METHODS):
    meds = []
    iqr_lo, iqr_hi = [], []
    for ki in range(len(KAPPAS)):
        vals = rq[mi, di_cost, ki, :, ITER]
        conv_mask = rq[mi, di_cost, ki, :, CONV] > 0.5
        v = vals[conv_mask]
        if len(v) > 0:
            meds.append(np.median(v))
            iqr_lo.append(np.percentile(v, 25))
            iqr_hi.append(np.percentile(v, 75))
        else:
            meds.append(0)
            iqr_lo.append(0)
            iqr_hi.append(0)
    meds = np.array(meds)
    lo_err = meds - np.array(iqr_lo)
    hi_err = np.array(iqr_hi) - meds
    pos = positions_base + (mi - 1) * width
    ax.bar(pos, meds, width * 0.85, color=METHOD_COLORS[method], alpha=0.7,
           label=method, yerr=[lo_err, hi_err], capsize=3, error_kw={"lw": 0.8})
ax.set_xticks(positions_base)
ax.set_xticklabels([str(k) for k in KAPPAS])
ax.set_xlabel(r"$\kappa$")
ax.set_ylabel("Itérations (médiane)")
ax.set_yscale("log")
ax.set_title(f"Nombre d'itérations ($n={DIMS[di_cost]}$)")
ax.legend(fontsize=9)
ax.grid(True, which="both", alpha=0.3, ls="--", axis="y")

ax = axes[1]
for mi, method in enumerate(METHODS):
    meds = []
    iqr_lo, iqr_hi = [], []
    for ki in range(len(KAPPAS)):
        vals = rq[mi, di_cost, ki, :, FEVAL]
        conv_mask = rq[mi, di_cost, ki, :, CONV] > 0.5
        v = vals[conv_mask]
        if len(v) > 0:
            meds.append(np.median(v))
            iqr_lo.append(np.percentile(v, 25))
            iqr_hi.append(np.percentile(v, 75))
        else:
            meds.append(0)
            iqr_lo.append(0)
            iqr_hi.append(0)
    meds = np.array(meds)
    lo_err = meds - np.array(iqr_lo)
    hi_err = np.array(iqr_hi) - meds
    lo_err = np.maximum(lo_err, 0)
    pos = positions_base + (mi - 1) * width
    ax.bar(pos, meds, width * 0.85, color=METHOD_COLORS[method], alpha=0.7,
           label=method, yerr=[lo_err, hi_err], capsize=3, error_kw={"lw": 0.8})
ax.set_xticks(positions_base)
ax.set_xticklabels([str(k) for k in KAPPAS])
ax.set_xlabel(r"$\kappa$")
ax.set_ylabel("Évaluations de $f$ (médiane)")
ax.set_yscale("log")
ax.set_title(f"Évaluations de $f$ ($n={DIMS[di_cost]}$)")
ax.legend(fontsize=9)
ax.grid(True, which="both", alpha=0.3, ls="--", axis="y")

fig.suptitle("Itérations et évaluations de $f$", fontsize=14, weight="bold")
fig.savefig(f"{OUT}/fig5_cost.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
if "metrics_rosenbrock" in results_gen_dict:
    res_rosen = results_gen_dict["metrics_rosenbrock"]
else:
    res_rosen = results_gen_dict.get("metrics_rosenbrock")

if "traj_rosenbrock" in traj_gen_dict:
    traj_rosen = traj_gen_dict["traj_rosenbrock"]
else:
    traj_rosen = traj_gen_dict.get("traj_rosenbrock")

fig, axes = plt.subplots(1, len(ROSEN_GAMMAS),
                         figsize=(5 * len(ROSEN_GAMMAS), 4.5),
                         constrained_layout=True)

for gi, gamma in enumerate(ROSEN_GAMMAS):
    ax = axes[gi]
    for mi, method in enumerate(GEN_METHODS):
        if traj_rosen is None:
            continue
        trajs = traj_rosen[mi, gi]
        lens = (~np.isnan(trajs)).sum(axis=1)
        max_common = int(np.median(lens))
        if max_common < 2:
            continue
        clipped = trajs[:, :max_common]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            med = np.nanmedian(clipped, axis=0)
            q25 = np.nanpercentile(clipped, 25, axis=0)
            q75 = np.nanpercentile(clipped, 75, axis=0)
        iters = np.arange(max_common)
        mask = med > 0
        ax.semilogy(iters[mask], med[mask], color=METHOD_COLORS[method],
                    ls=METHOD_LS[method], lw=1.8, label=method)
        ax.fill_between(iters[mask], q25[mask], q75[mask],
                        color=METHOD_COLORS[method], alpha=0.12)

    ax.set_title(rf"$\gamma = {gamma}$")
    ax.set_xlabel("Itérations")
    if gi == 0:
        ax.set_ylabel(r"$\|x_k - x^\star\|_2$")
        ax.legend(fontsize=9)
    ax.grid(True, which="major", alpha=0.3, ls="--")

fig.suptitle(r"Fonction de Rosenbrock : CG-PR et BFGS",
             fontsize=13, weight="bold")
fig.savefig(f"{OUT}/fig6_rosenbrock.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
TITLE_MAP = {"rosenbrock": "Fonction de Rosenbrock", "multitrous2": "Multitrous"}

for fi, fname in enumerate(FUNC_NAMES):
    ax = axes[fi]
    gammas = FUNC_CONFIGS[fname]["gammas"]
    res = results_gen_dict[f"metrics_{fname}"]
    n_g = len(gammas)
    width = 0.35
    positions_base = np.arange(n_g) * 2.0

    for mi, method in enumerate(GEN_METHODS):
        data = []
        for gi in range(n_g):
            vals = res[mi, gi, :, G_ITER]
            conv_mask = res[mi, gi, :, G_CONV] > 0.5
            v = vals[conv_mask]
            data.append(v if len(v) > 0 else np.array([np.nan]))
        pos = positions_base + mi * (width + 0.05)
        bp = ax.boxplot(data, positions=pos, widths=width,
                        patch_artist=True, showfliers=False)
        for patch in bp["boxes"]:
            patch.set_facecolor(METHOD_COLORS[method])
            patch.set_alpha(0.7)
        for element in ["whiskers", "caps", "medians"]:
            for line in bp[element]:
                line.set_color("black")
                line.set_linewidth(0.8)

    ax.set_xticks(positions_base + width / 2)
    ax.set_xticklabels([rf"$\gamma={g}$" for g in gammas])
    ax.set_ylabel("Itérations")
    ax.set_title(TITLE_MAP[fname])
    ax.set_yscale("log")
    ax.grid(True, which="both", alpha=0.3, ls="--")
    if fi == 0:
        handles = [plt.Line2D([0], [0], color=METHOD_COLORS[m], lw=6, alpha=0.7)
                   for m in GEN_METHODS]
        ax.legend(handles, GEN_METHODS, loc="upper left", fontsize=10)

    for mi, method in enumerate(GEN_METHODS):
        rates = [f"{res[mi, gi, :, G_CONV].mean():.0%}" for gi in range(n_g)]
        print(f"  {fname} {method}: conv rates = {rates}")

fig.suptitle("Régime général : itérations (exécutions convergées)", fontsize=14, weight="bold")
fig.savefig(f"{OUT}/fig7_general_iters.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
print("=" * 80)
print("REGIME QUADRATIQUE (n=10)")
print("=" * 80)
di = DIMS.index(10)
print(f"{'Method':<10} {'kappa':>6} | {'med iter':>9} {'med feval':>10} "
      f"{'med err':>10} {'conv rate':>10}")
print("-" * 70)
for mi, method in enumerate(METHODS):
    for ki, kappa in enumerate(KAPPAS):
        conv_mask = rq[mi, di, ki, :, CONV] > 0.5
        n_conv = conv_mask.sum()
        if n_conv > 0:
            med_it = np.median(rq[mi, di, ki, conv_mask, ITER])
            med_fv = np.median(rq[mi, di, ki, conv_mask, FEVAL])
            med_err = np.median(rq[mi, di, ki, conv_mask, ERR])
            rate = n_conv / N_SEEDS
            print(f"{method:<10} {kappa:>6} | {med_it:>9.0f} {med_fv:>10.0f} "
                  f"{med_err:>10.2e} {rate:>10.0%}")
        else:
            print(f"{method:<10} {kappa:>6} | {'---':>9} {'---':>10} "
                  f"{'---':>10} {'0%':>10}")

print()
print("=" * 80)
print("REGIME GENERAL")
print("=" * 80)
for fname in FUNC_NAMES:
    res = results_gen_dict[f"metrics_{fname}"]
    gammas = FUNC_CONFIGS[fname]["gammas"]
    print(f"\n--- {fname} ---")
    print(f"{'Method':<10} {'gamma':>6} | {'med iter':>9} {'med feval':>10} {'conv rate':>10}")
    print("-" * 55)
    for mi, method in enumerate(GEN_METHODS):
        for gi, gamma in enumerate(gammas):
            conv_mask = res[mi, gi, :, G_CONV] > 0.5
            n_conv = conv_mask.sum()
            if n_conv > 0:
                med_it = np.median(res[mi, gi, conv_mask, G_ITER])
                med_fv = np.median(res[mi, gi, conv_mask, G_FEVAL])
                rate = n_conv / N_X0
                print(f"{method:<10} {gamma:>6} | {med_it:>9.0f} {med_fv:>10.0f} {rate:>10.0%}")
            else:
                print(f"{method:<10} {gamma:>6} | {'---':>9} {'---':>10} {'0%':>10}")